# 05 - Statistical Analysis

## Objective

This notebook validates the project research questions using statistical hypothesis testing on the cleaned 2025 Airline On-Time Performance dataset.

The analysis defines null and alternative hypotheses, selects appropriate statistical tests, checks key assumptions, computes p-values and effect sizes, and documents whether each research hypothesis is supported.

Statistical evidence generated here supports the final report and the downstream predictive and prescriptive stages of the project.

#### Load project configuration

In [0]:
# Load the project configuration

from __future__ import annotations

from config import project_config as cfg

print("Project configuration loaded successfully.")
print(f"Source table: {cfg.CLEAN_TABLE}")
print(f"Results table: {cfg.STATISTICAL_RESULTS_TABLE}")
print(f"Significance level: {cfg.STATISTICAL_SIGNIFICANCE_ALPHA}")


#### Load and validate the cleaned dataset

The statistical analysis uses the managed `flights_clean` table produced by the data-cleaning notebook. Only completed operational flights are retained for hypothesis testing.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-cleaning notebook before continuing."
        )


require_table(cfg.CLEAN_TABLE)

df_clean: DataFrame = spark.table(cfg.CLEAN_TABLE)

clean_row_count = df_clean.count()
missing_columns = sorted(
    set(cfg.FEATURE_INPUT_COLUMNS) - set(df_clean.columns)
)

if missing_columns:
    raise ValueError(
        "Statistical-analysis validation failed. "
        f"Missing required columns: {missing_columns}"
    )

print("Cleaned dataset loaded and validated successfully.")
print(f"Total records: {clean_row_count:,}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


#### Create the analysis population

Cancelled and diverted flights are excluded because the target variable represents arrival delay for completed flights. Schedule-time diagnostic fields are derived for statistical testing.

In [0]:
df_analysis = (
    df_clean
    .filter(
        (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    )
    .withColumn(
        cfg.DEP_HOUR_COLUMN,
        F.floor(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN) / 100).cast("int"),
    )
    .withColumn(
        cfg.IS_WEEKEND_COLUMN,
        F.when(
            F.col(cfg.DAY_OF_WEEK_COLUMN).isin(*cfg.WEEKEND_DAYS),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        cfg.SEASON_COLUMN,
        F.when(
            F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Winter"]),
            "Winter",
        )
        .when(
            F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Spring"]),
            "Spring",
        )
        .when(
            F.col(cfg.MONTH_COLUMN).isin(*cfg.SEASON_MONTH_GROUPS["Summer"]),
            "Summer",
        )
        .otherwise("Fall"),
    )
    .withColumn(
        cfg.TIME_OF_DAY_COLUMN,
        F.when(F.col(cfg.DEP_HOUR_COLUMN).between(0, 5), "Overnight")
        .when(F.col(cfg.DEP_HOUR_COLUMN).between(6, 11), "Morning")
        .when(F.col(cfg.DEP_HOUR_COLUMN).between(12, 16), "Afternoon")
        .when(F.col(cfg.DEP_HOUR_COLUMN).between(17, 20), "Evening")
        .otherwise("Night"),
    )
)

analysis_row_count = df_analysis.count()

analysis_summary = spark.createDataFrame(
    [
        (
            clean_row_count,
            analysis_row_count,
            clean_row_count - analysis_row_count,
        )
    ],
    schema="CLEAN_ROWS long, ANALYSIS_ROWS long, EXCLUDED_ROWS long",
)

display(analysis_summary)


#### Define research questions and hypotheses

The research design follows the project proposal and final report.

| ID | Research Question |
|---|---|
| RQ1 | Can flight delays be accurately predicted before departure using operational flight information? |
| RQ2 | Which operational factors contribute the most to flight delays? |
| RQ3 | Do certain airlines and airports consistently experience higher delay rates than others? |
| RQ4 | Can prescriptive analytics improve operational decision-making by prioritizing high-risk flights under limited resources? |
| RQ5 | Can Explainable Artificial Intelligence (SHAP) improve the interpretability of flight delay predictions for airline operations? |

| Hypothesis | Null Hypothesis (H0) | Alternative Hypothesis (H1) |
|---|---|---|
| H1 | Operational flight variables do not significantly predict whether a flight will be delayed. | Operational flight variables significantly predict whether a flight will be delayed. |
| H2 | There is no statistically significant relationship between operational factors and arrival delays. | At least one operational factor has a statistically significant relationship with arrival delays. |
| H3 | There is no statistically significant difference in delay rates among airlines and airports. | There is a statistically significant difference in delay rates among airlines and airports. |
| H4 | Optimization-based prioritization does not improve operational efficiency compared with a non-prioritized approach. | Optimization-based prioritization significantly improves operational efficiency compared with a non-prioritized approach. |

#### Create the statistical analysis sample

Because the full dataset contains millions of records, a reproducible stratified sample is used for inferential tests executed in Python. Aggregated Spark summaries are used where appropriate.

In [0]:
from utils import statistical_analysis as stat


analysis_columns = [
    cfg.TARGET_COLUMN,
    cfg.DISTANCE_COLUMN,
    cfg.MONTH_COLUMN,
    cfg.DAY_OF_WEEK_COLUMN,
    cfg.AIRLINE_COLUMN,
    cfg.ORIGIN_COLUMN,
    cfg.DESTINATION_COLUMN,
    cfg.SEASON_COLUMN,
    cfg.TIME_OF_DAY_COLUMN,
    cfg.IS_WEEKEND_COLUMN,
    cfg.DEP_HOUR_COLUMN,
]

analysis_sample = (
    df_analysis
    .select(*analysis_columns)
    .sample(
        withReplacement=False,
        fraction=cfg.STATISTICAL_SAMPLE_FRACTION,
        seed=cfg.RANDOM_SEED,
    )
)

analysis_pdf = analysis_sample.toPandas()

print(f"Analysis sample rows: {len(analysis_pdf):,}")
print(
    f"Observed delay rate: "
    f"{analysis_pdf[cfg.TARGET_COLUMN].mean():.4f}"
)

display(analysis_sample.limit(5))


#### Helper functions for grouped categorical tests

High-cardinality categorical variables are grouped into the most frequent categories plus an `OTHER` bucket so chi-square expected frequencies remain valid.

In [0]:
import pandas as pd


def bucket_top_categories(
    frame,
    column_name: str,
    top_n: int,
) -> str:
    grouped_column = f"{column_name}_GROUPED"
    top_values = (
        frame[column_name]
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )
    frame[grouped_column] = frame[column_name].apply(
        lambda value: value if value in top_values else "OTHER"
    )
    return grouped_column


def run_categorical_test(
    *,
    research_question: str,
    hypothesis_id: str,
    null_hypothesis: str,
    alternative_hypothesis: str,
    frame,
    factor_column: str,
    top_n: int | None = None,
) -> dict:
    working_frame = frame.copy()
    test_column = factor_column

    if top_n is not None:
        test_column = bucket_top_categories(
            working_frame,
            factor_column,
            top_n,
        )

    contingency_table = pd.crosstab(
        working_frame[test_column],
        working_frame[cfg.TARGET_COLUMN],
    )
    test_output = stat.chi_square_independence(
        contingency_table,
        min_expected_frequency=cfg.STATISTICAL_MIN_EXPECTED_FREQUENCY,
    )
    return stat.build_test_result_row(
        research_question=research_question,
        hypothesis_id=hypothesis_id,
        null_hypothesis=null_hypothesis,
        alternative_hypothesis=alternative_hypothesis,
        factor=test_column,
        test_output=test_output,
        alpha=cfg.STATISTICAL_SIGNIFICANCE_ALPHA,
    )


#### RQ2 / H2 — Relationship between operational factors and arrival delays

This section tests whether schedule-time operational factors are statistically associated with the binary arrival-delay target.

Tests applied:

- Pearson correlation between flight distance and arrival delay
- Chi-square tests for categorical schedule factors
- Welch's t-test comparing distance between on-time and delayed flights

In [0]:
rq2_results = []

distance_test = stat.pearson_correlation(
    analysis_pdf[cfg.DISTANCE_COLUMN],
    analysis_pdf[cfg.TARGET_COLUMN],
)
rq2_results.append(
    stat.build_test_result_row(
        research_question="RQ2",
        hypothesis_id="H2",
        null_hypothesis="There is no statistically significant relationship between operational factors and arrival delays.",
        alternative_hypothesis="At least one operational factor has a statistically significant relationship with arrival delays.",
        factor=cfg.DISTANCE_COLUMN,
        test_output=distance_test,
        alpha=cfg.STATISTICAL_SIGNIFICANCE_ALPHA,
    )
)

categorical_factor_settings = [
    (cfg.MONTH_COLUMN, None),
    (cfg.DAY_OF_WEEK_COLUMN, None),
    (cfg.SEASON_COLUMN, None),
    (cfg.TIME_OF_DAY_COLUMN, None),
    (cfg.IS_WEEKEND_COLUMN, None),
    (cfg.AIRLINE_COLUMN, cfg.STATISTICAL_TOP_AIRLINES),
    (cfg.ORIGIN_COLUMN, cfg.STATISTICAL_TOP_AIRPORTS),
    (cfg.DESTINATION_COLUMN, cfg.STATISTICAL_TOP_DEST_AIRPORTS),
]

for factor_column, top_n in categorical_factor_settings:
    rq2_results.append(
        run_categorical_test(
            research_question="RQ2",
            hypothesis_id="H2",
            null_hypothesis="There is no statistically significant relationship between operational factors and arrival delays.",
            alternative_hypothesis="At least one operational factor has a statistically significant relationship with arrival delays.",
            frame=analysis_pdf,
            factor_column=factor_column,
            top_n=top_n,
        )
    )

distance_t_test = stat.independent_t_test(
    analysis_pdf.loc[
        analysis_pdf[cfg.TARGET_COLUMN] == 0,
        cfg.DISTANCE_COLUMN,
    ],
    analysis_pdf.loc[
        analysis_pdf[cfg.TARGET_COLUMN] == 1,
        cfg.DISTANCE_COLUMN,
    ],
)
rq2_results.append(
    stat.build_test_result_row(
        research_question="RQ2",
        hypothesis_id="H2",
        null_hypothesis="There is no statistically significant relationship between operational factors and arrival delays.",
        alternative_hypothesis="At least one operational factor has a statistically significant relationship with arrival delays.",
        factor=f"{cfg.DISTANCE_COLUMN} (delayed vs on-time)",
        test_output=distance_t_test,
        alpha=cfg.STATISTICAL_SIGNIFICANCE_ALPHA,
    )
)

rq2_results_df = pd.DataFrame(rq2_results)
display(spark.createDataFrame(rq2_results_df))


#### RQ2 / H2 — One-way ANOVA for departure hour

Departure hour is treated as an operational schedule factor with multiple groups. A one-way ANOVA compares mean delay outcomes across scheduled departure hours.

In [0]:
hour_groups = [
    analysis_pdf.loc[analysis_pdf[cfg.DEP_HOUR_COLUMN] == hour_value, cfg.TARGET_COLUMN]
    for hour_value in sorted(analysis_pdf[cfg.DEP_HOUR_COLUMN].dropna().unique())
]

hour_anova = stat.one_way_anova(hour_groups)
hour_anova_row = stat.build_test_result_row(
    research_question="RQ2",
    hypothesis_id="H2",
    null_hypothesis="There is no statistically significant relationship between operational factors and arrival delays.",
    alternative_hypothesis="At least one operational factor has a statistically significant relationship with arrival delays.",
    factor=cfg.DEP_HOUR_COLUMN,
    test_output=hour_anova,
    alpha=cfg.STATISTICAL_SIGNIFICANCE_ALPHA,
)

rq2_results.append(hour_anova_row)
rq2_results_df = pd.DataFrame(rq2_results)
display(spark.createDataFrame(pd.DataFrame([hour_anova_row])))


#### RQ3 / H3 — Differences in delay rates among airlines and airports

Chi-square tests evaluate whether delay outcomes differ significantly across airlines and airports.

In [0]:
rq3_results = [
    run_categorical_test(
        research_question="RQ3",
        hypothesis_id="H3",
        null_hypothesis="There is no statistically significant difference in delay rates among airlines and airports.",
        alternative_hypothesis="There is a statistically significant difference in delay rates among airlines and airports.",
        frame=analysis_pdf,
        factor_column=cfg.AIRLINE_COLUMN,
        top_n=cfg.STATISTICAL_TOP_AIRLINES,
    ),
    run_categorical_test(
        research_question="RQ3",
        hypothesis_id="H3",
        null_hypothesis="There is no statistically significant difference in delay rates among airlines and airports.",
        alternative_hypothesis="There is a statistically significant difference in delay rates among airlines and airports.",
        frame=analysis_pdf,
        factor_column=cfg.ORIGIN_COLUMN,
        top_n=cfg.STATISTICAL_TOP_AIRPORTS,
    ),
    run_categorical_test(
        research_question="RQ3",
        hypothesis_id="H3",
        null_hypothesis="There is no statistically significant difference in delay rates among airlines and airports.",
        alternative_hypothesis="There is a statistically significant difference in delay rates among airlines and airports.",
        frame=analysis_pdf,
        factor_column=cfg.DESTINATION_COLUMN,
        top_n=cfg.STATISTICAL_TOP_DEST_AIRPORTS,
    ),
]

rq3_results_df = pd.DataFrame(rq3_results)
display(spark.createDataFrame(rq3_results_df))


#### RQ1 / H1 — Schedule-time association with delay outcomes

RQ1 is ultimately validated through predictive modeling in the model-training notebook. At this stage, chi-square tests on schedule-time variables provide statistical evidence that pre-departure operational information is associated with delay outcomes.

In [0]:
rq1_results = [
    run_categorical_test(
        research_question="RQ1",
        hypothesis_id="H1",
        null_hypothesis="Operational flight variables do not significantly predict whether a flight will be delayed.",
        alternative_hypothesis="Operational flight variables significantly predict whether a flight will be delayed.",
        frame=analysis_pdf,
        factor_column=cfg.TIME_OF_DAY_COLUMN,
    ),
    run_categorical_test(
        research_question="RQ1",
        hypothesis_id="H1",
        null_hypothesis="Operational flight variables do not significantly predict whether a flight will be delayed.",
        alternative_hypothesis="Operational flight variables significantly predict whether a flight will be delayed.",
        frame=analysis_pdf,
        factor_column=cfg.SEASON_COLUMN,
    ),
    run_categorical_test(
        research_question="RQ1",
        hypothesis_id="H1",
        null_hypothesis="Operational flight variables do not significantly predict whether a flight will be delayed.",
        alternative_hypothesis="Operational flight variables significantly predict whether a flight will be delayed.",
        frame=analysis_pdf,
        factor_column=cfg.IS_WEEKEND_COLUMN,
    ),
]

rq1_results_df = pd.DataFrame(rq1_results)
display(spark.createDataFrame(rq1_results_df))


#### RQ4 and RQ5 — Deferred statistical validation

- **RQ4 / H4** will be evaluated in the operational-prioritization stage using optimization experiments that compare prioritized versus non-prioritized review strategies.
- **RQ5** will be evaluated in the SHAP explainability stage through model-interpretability analysis rather than a classical inferential hypothesis test.

These research questions remain part of the project design but are not statistically tested in this notebook.

#### Consolidate statistical evidence

All completed hypothesis tests are combined into a single results table for reporting and downstream dashboard use.

In [0]:
statistical_results_pdf = pd.concat(
    [rq2_results_df, rq3_results_df, rq1_results_df],
    ignore_index=True,
)

statistical_results_pdf["interpretation"] = statistical_results_pdf.apply(
    lambda row: (
        f"{row['decision']} at alpha={row['alpha']:.2f}; "
        f"effect size ({row['effect_size_label']})={row['effect_size']:.4f}"
    ),
    axis=1,
)

statistical_results_df = spark.createDataFrame(statistical_results_pdf)
display(statistical_results_df.orderBy("research_question", "factor"))


#### Research question validation summary

Use the consolidated results table to determine whether each research question is supported:

- **RQ2:** Supported if at least one operational factor shows a statistically significant association with `ARR_DEL15`.
- **RQ3:** Supported if airline or airport grouping shows a statistically significant difference in delay outcomes.
- **RQ1:** Supported statistically when schedule-time variables show significant association; final predictive confirmation occurs in model training.
- **RQ4 / RQ5:** Validated in later project stages.

In [0]:
validation_summary = (
    statistical_results_pdf
    .groupby("research_question", as_index=False)
    .agg(
        tests_run=("factor", "count"),
        significant_tests=("decision", lambda s: (s == "Reject H0").sum()),
        supported=("decision", lambda s: "Yes" if (s == "Reject H0").any() else "No"),
    )
)

display(spark.createDataFrame(validation_summary))


#### Save statistical analysis results

The consolidated results are saved to the processed layer and registered as a managed Delta table.

In [0]:
(
    statistical_results_df.write
    .format("delta")
    .mode("overwrite")
    .save(cfg.STATISTICAL_RESULTS_PATH)
)

(
    statistical_results_df.writeTo(cfg.STATISTICAL_RESULTS_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Statistical analysis results saved successfully.")
print(f"Delta path: {cfg.STATISTICAL_RESULTS_PATH}")
print(f"Table: {cfg.STATISTICAL_RESULTS_TABLE}")
print(f"Total tests stored: {statistical_results_df.count():,}")
